# Batch cache NeuroMorpho neurons, then batch compare with `braincell`

This notebook is organized as two direct steps:

1. use a typed `NeuroMorphoQuery` to batch search NeuroMorpho neurons
2. cache every neuron under `examples/multi_compartment/data/neuromorpho/<neuron_id>/` via `NeuroMorphoClient.download`
3. store `metadata.json` with neuron metadata, measurement, and download links
4. scan cached folders with `NeuroMorphoCache`, load each standardized `SWC`, and compare against the cached NeuroMorpho measurement

For one-shot use cases that just need a `Morphology` object, prefer the Tier-1 helper:

```python
from braincell import load_neuromorpho

morph = load_neuromorpho(10047)              # one call → parsed Morphology
morph = Morphology.from_neuromorpho(10047)   # equivalent classmethod entry point
```

This notebook uses the Tier-2 client (`NeuroMorphoClient`) and the cache object (`NeuroMorphoCache`) so it can iterate over many neurons and validate `braincell`'s metric computations against the upstream measurement records.


In [ ]:
import json
import os
import sys
from pathlib import Path

import brainunit as u

os.environ.setdefault("JAX_PLATFORMS", "cpu")


def find_repo_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "braincell").exists() and (candidate / "examples").exists():
            return candidate
    raise RuntimeError("Run this notebook from the repository root or a subdirectory inside it.")


repo_root = find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import braincell
from braincell import Morphology, load_neuromorpho
from braincell.io import NeuroMorphoCache, NeuroMorphoClient, NeuroMorphoQuery

print("braincell version:", braincell.__version__)
print("repo_root:", repo_root)


In [2]:
EXACT_MATCH_KEYS = ("n_stems", "n_branch", "n_bifs")
FLOAT_THRESHOLDS = {
    "length": 0.001,
    "surface": 0.01,
    "volume": 0.01,
    "pathDistance": 0.001,
    "eucDistance": 0.001,
    "soma_Surface": 0.10,
}
REQUIRED_MEASUREMENT_KEYS = EXACT_MATCH_KEYS + tuple(FLOAT_THRESHOLDS)


def missing_measurement_keys(measurement):
    return [key for key in REQUIRED_MEASUREMENT_KEYS if measurement.get(key) is None]


def has_soma_branch(tree):
    return any(branch.type == "soma" for branch in tree.branches)


def compute_braincell_metrics(tree):
    soma_branches = [branch for branch in tree.branches if branch.type == "soma"]
    non_soma_branches = [branch for branch in tree.branches if branch.type != "soma"]
    return {
        "n_stems": int(tree.n_stems),
        "n_branch": int(len(non_soma_branches)),
        "n_bifs": int(sum(branch.n_children >= 2 for branch in non_soma_branches)),
        "length": float(sum(branch.length.to_decimal(u.um) for branch in non_soma_branches)),
        "surface": float(sum(branch.area.to_decimal(u.um ** 2) for branch in non_soma_branches)),
        "volume": float(sum(branch.volume.to_decimal(u.um ** 3) for branch in non_soma_branches)),
        "pathDistance": float(tree.max_path_distance_excluding_soma.to_decimal(u.um)),
        "eucDistance": float(tree.max_euclidean_distance_excluding_soma.to_decimal(u.um)),
        "soma_Surface": float(sum(branch.area.to_decimal(u.um ** 2) for branch in soma_branches)),
    }


def compute_neuromorpho_metrics(measurement):
    return {
        "n_stems": int(round(float(measurement["n_stems"]))),
        "n_branch": int(round(float(measurement["n_branch"]))),
        "n_bifs": int(round(float(measurement["n_bifs"]))),
        "length": float(measurement["length"]),
        "surface": float(measurement["surface"]),
        "volume": float(measurement["volume"]),
        "pathDistance": float(measurement["pathDistance"]),
        "eucDistance": float(measurement["eucDistance"]),
        "soma_Surface": float(measurement["soma_Surface"]),
    }


def compare_tree_with_measurement(tree, measurement):
    braincell_metrics = compute_braincell_metrics(tree)
    neuromorpho_metrics = compute_neuromorpho_metrics(measurement)
    metric_results = {}

    for key, target in neuromorpho_metrics.items():
        actual = braincell_metrics[key]
        if key in EXACT_MATCH_KEYS:
            abs_diff = abs(actual - target)
            rel_diff = None
            passed = actual == target
        else:
            abs_diff = abs(actual - target)
            if abs(target) <= 1e-12:
                rel_diff = 0.0 if abs_diff <= 1e-12 else None
                passed = abs_diff <= 1e-12
            else:
                rel_diff = abs_diff / abs(target)
                passed = rel_diff < FLOAT_THRESHOLDS[key]

        metric_results[key] = {
            "braincell": actual,
            "neuromorpho": target,
            "abs_diff": abs_diff,
            "rel_diff": rel_diff,
            "threshold": FLOAT_THRESHOLDS.get(key),
            "pass": passed,
        }

    non_soma_float_keys = [
        key for key in metric_results
        if key not in EXACT_MATCH_KEYS and key != "soma_Surface"
    ]
    non_soma_rel_diffs = [
        metric_results[key]["rel_diff"]
        for key in non_soma_float_keys
        if metric_results[key]["rel_diff"] is not None
    ]

    integer_pass = all(metric_results[key]["pass"] for key in EXACT_MATCH_KEYS)
    non_soma_float_pass = all(metric_results[key]["pass"] for key in non_soma_float_keys)
    soma_surface_pass = metric_results["soma_Surface"]["pass"]

    return {
        "braincell_metrics": braincell_metrics,
        "neuromorpho_metrics": neuromorpho_metrics,
        "metric_results": metric_results,
        "integer_pass": integer_pass,
        "non_soma_float_pass": non_soma_float_pass,
        "soma_surface_pass": soma_surface_pass,
        "overall_pass": integer_pass and non_soma_float_pass and soma_surface_pass,
        "non_soma_max_rel_diff": max(non_soma_rel_diffs) if non_soma_rel_diffs else 0.0,
        "soma_surface_rel_diff": metric_results["soma_Surface"]["rel_diff"],
    }


## 1. Batch query and cache neurons

Use a typed `NeuroMorphoQuery` (or a raw Solr `q`/`fq` string) and `client.iter_search(...)` to lazily stream every matching neuron, then download each into:

`examples/multi_compartment/data/neuromorpho/<neuron_id>/`

Each neuron folder will contain:

- the standardized `SWC`, the original file, or both
- `metadata.json`
- the full NeuroMorpho measurement payload and links needed to trace the neuron later


In [ ]:
query = NeuroMorphoQuery(
    species="mouse",
    brain_region="cerebellum",
)
size = 100
page_start = 5
limit = 500  # set to None to stream every match
sort = "neuron_id,asc"
download_mode = "standard"  # choose from: "standard", "original", "both"
overwrite = False
output_root = repo_root / "examples" / "multi_compartment" / "data" / "neuromorpho"

client = NeuroMorphoClient(cache_dir=output_root)
neurons = list(
    client.iter_search(
        query,
        size=size,
        start_page=page_start,
        limit=limit,
        sort=sort,
    )
)
if not neurons:
    raise RuntimeError(
        "No NeuroMorpho records matched the current query. Relax the filters and run again."
    )

print("query.q:", query.to_q())
print("query.fq:", query.to_fq())
print("size:", size)
print("page_start:", page_start)
print("limit:", limit)
print("download_mode:", download_mode)
print("output_root:", output_root)
print()
print("matched_neurons:", len(neurons))
print()
for neuron in neurons[:10]:
    print(f"id={neuron.neuron_id} name={neuron.neuron_name}")
    print("  archive:", neuron.archive)
    print("  brain_region:", neuron.brain_region)
    print("  original_format:", neuron.original_format)
    print()
if len(neurons) > 10:
    print(f"... and {len(neurons) - 10} more")


In [ ]:
cache = NeuroMorphoCache(output_root)
download_records = []
for neuron in neurons:
    record = client.download(
        neuron,
        output_dir=output_root,
        mode=download_mode,
        overwrite=overwrite,
    )
    download_records.append(record)

    print(f"cached neuron_id={neuron.neuron_id} name={neuron.neuron_name}")
    print("  folder:", record.folder)
    print("  metadata:", record.metadata_path)
    for item in record.download_items:
        print(f"  {item.kind}: {item.filename}")
        print("    url:", item.url)
        print("    path:", item.path)
        print("    downloaded_now:", item.downloaded_now)
        print("    reason:", item.reason)
    print()

if download_records:
    sample_neuron = neurons[0]
    sample_metadata = cache.metadata(sample_neuron.neuron_id)
    sample_detail = client.describe(sample_neuron)
    print("sample_metadata_file:", download_records[0].metadata_path)
    print("sample_measurement_url:", sample_metadata.get("measurement_url"))
    print("sample_thumbnail_url:", sample_detail.urls.thumbnail)
    print("sample_standard_swc_url:", sample_detail.urls.standard_swc)
    print("sample_original_file_url:", sample_detail.urls.original_file)
    print("sample_measurement (raw upstream payload):")
    print(sample_metadata["measurement"])
    print()
    print("typed measurement (snake_case attributes):")
    typed_meas = sample_detail.measurement
    print("  length:", typed_meas.length)
    print("  surface:", typed_meas.surface)
    print("  path_distance:", typed_meas.path_distance)
    print("  euclidean_distance:", typed_meas.euclidean_distance)
    print("  soma_surface:", typed_meas.soma_surface)


## 2. Scan cached folders and batch compare

This step does not re-query NeuroMorpho. It uses `NeuroMorphoCache` to enumerate neurons cached on disk under `examples/multi_compartment/data/neuromorpho/`, loads each cached standardized `SWC` via `cache.load(...)`, and compares the resulting `Morphology` against the cached upstream `measurement` stored in `metadata.json`.


In [ ]:
cache = NeuroMorphoCache(output_root)
cached_ids = cache.list_neurons()

comparison_runs = []
summary = {
    "scanned": 0,
    "compared": 0,
    "skipped": 0,
    "passed": 0,
    "failed": 0,
    "errors": 0,
}

if not cached_ids:
    print("No cached neuron folders were found under", output_root)
else:
    for neuron_id in cached_ids:
        summary["scanned"] += 1
        folder = cache.layout.neuron_dir(neuron_id)

        try:
            metadata = cache.metadata(neuron_id)
        except FileNotFoundError:
            summary["skipped"] += 1
            print(f"[skip] {folder.name}: missing metadata.json")
            continue
        except Exception as exc:
            summary["errors"] += 1
            print(f"[error] {folder.name}: failed to read metadata.json: {exc}")
            continue

        measurement = metadata.get("measurement")
        if not measurement:
            summary["skipped"] += 1
            print(f"[skip] {folder.name}: metadata.json does not contain measurement")
            continue

        missing_keys = missing_measurement_keys(measurement)
        if missing_keys:
            summary["skipped"] += 1
            print(f"[skip] {folder.name}: incomplete measurement for keys {missing_keys}")
            continue

        swc_path = cache.standard_swc_path(neuron_id)
        if swc_path is None or not swc_path.exists():
            summary["skipped"] += 1
            print(f"[skip] {folder.name}: no standardized SWC found")
            continue

        try:
            tree, report = Morphology.from_swc(swc_path, return_report=True, mode="neuromorpho")
        except Exception as exc:
            summary["errors"] += 1
            print(f"[error] {folder.name}: failed to load {swc_path}: {exc}")
            continue

        if not has_soma_branch(tree):
            summary["skipped"] += 1
            print(f"[skip] {folder.name}: no soma branches in {swc_path}")
            continue

        try:
            comparison = compare_tree_with_measurement(tree, measurement)
        except Exception as exc:
            summary["errors"] += 1
            print(f"[error] {folder.name}: failed to compare {swc_path}: {exc}")
            continue

        summary["compared"] += 1
        if comparison["overall_pass"]:
            summary["passed"] += 1
        else:
            summary["failed"] += 1

        comparison_runs.append(
            {
                "folder": str(folder),
                "neuron_id": metadata.get("neuron_id"),
                "neuron_name": metadata.get("neuron_name"),
                "swc_path": str(swc_path),
                "comparison": comparison,
            }
        )

        print("=" * 80)
        print(f"neuron_id: {metadata.get('neuron_id')}")
        print(f"neuron_name: {metadata.get('neuron_name')}")
        print(f"folder: {folder}")
        print(f"swc_path: {swc_path}")
        print()

        for key, result in comparison["metric_results"].items():
            rel_text = "n/a" if result["rel_diff"] is None else f"{result['rel_diff']:.6%}"
            print(
                f"{key}: braincell={result['braincell']:.6f}, neuromorpho={result['neuromorpho']:.6f}, "
                f"abs_diff={result['abs_diff']:.6f}, rel_diff={rel_text}, pass={result['pass']}"
            )

        print()
        print(f"integer_pass: {comparison['integer_pass']}")
        print(f"non_soma_max_rel_diff: {comparison['non_soma_max_rel_diff']:.6%}")
        print(
            "soma_surface_rel_diff:",
            "n/a" if comparison["soma_surface_rel_diff"] is None else f"{comparison['soma_surface_rel_diff']:.6%}",
        )
        print(f"non_soma_float_pass: {comparison['non_soma_float_pass']}")
        print(f"soma_surface_pass: {comparison['soma_surface_pass']}")
        print(f"overall_pass: {comparison['overall_pass']}")

    print("=" * 80)
    print("scan summary")
    print("scanned:", summary["scanned"])
    print("compared:", summary["compared"])
    print("skipped:", summary["skipped"])
    print("passed:", summary["passed"])
    print("failed:", summary["failed"])
    print("errors:", summary["errors"])

comparison_runs
